# 09 Reproducible Training and Inference Pipelines

After refactoring, we can run the same workflow without copying notebook cells. This notebook introduces two explicit pipelines:

- a training pipeline
- an inference pipeline

The important idea is that inference must use the same preprocessing logic that was fitted during training.

## 1. Why We Need Two Pipelines

During training, we have historical data with known labels. The training pipeline can clean the data, create features, split the data, fit preprocessing steps, train a model, evaluate it, and save artifacts.

During inference, we receive new observations. Their labels are not known yet. The inference pipeline must create the same input features and then apply the already fitted preprocessor and model.

This is why the fitted preprocessor is an artifact. It contains the learned imputation values, scaling parameters, and one-hot encoding structure from the training data.

## 2. Artifact Structure

The training pipeline saves two local artifacts with the same timestamp and logs them to the MLflow run:

```text
models/
  <timestamp>_preprocessor.joblib
  <timestamp>_XGBClassifier_model.joblib
```

The model alone is not enough for inference. New raw observations must first go through the saved preprocessor. The `models/` directory is ignored by Git because these files are generated artifacts. MLflow keeps a run record with the metrics and copies of the model and preprocessor artifacts, while the local `models/` paths remain convenient for the next inference step.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root. Please run this notebook inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, REPORTS_DIR
from src.pipeline import run_inference_pipeline, run_training_pipeline

MODEL_RESULTS_DIR = REPORTS_DIR / "model_results"

print(PROJECT_ROOT)

2026-06-18 11:51:58.164 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master


/home/cbaldermann/Envs/ads_II/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master


## 3. Choose the Model Parameters for Training

The training pipeline should train one defined model configuration. It should not silently invent new hyperparameters.

There are several practical ways to provide the selected parameters:

- **Manual option:** write the parameters directly in the training notebook or script. This is simple, but easy to copy incorrectly.
- **Artifact option used here:** load the selected parameters from the JSON file written by Notebook 05. This makes the handoff from model selection to training explicit.
- **MLflow-centered option:** query the best MLflow run and load the parameters from that run. This is common in a more mature setup, but it requires that hyperparameter optimization runs were logged to MLflow in the first place.

In this project step, we use the JSON artifact because it is transparent and keeps Notebook 09 independent from manual copy/paste.

In [2]:
selected_params_path = MODEL_RESULTS_DIR / "05_selected_xgboost_params.json"
with selected_params_path.open("r", encoding="utf-8") as file:
    selected_model_config = json.load(file)

selected_xgb_params = selected_model_config["params"]
selected_model_name = selected_model_config["model_name"]

print("Selected model:", selected_model_name)
print("Selected because:", selected_model_config["selected_by"])
selected_xgb_params

Selected model: XGBoost Bayesian Search
Selected because: highest holdout F1 in xgboost_comparison_df


{'n_estimators': 156,
 'learning_rate': 0.05082341959721458,
 'max_depth': 2,
 'min_child_weight': 7,
 'subsample': 0.6760927252879199,
 'colsample_bytree': 0.9954104278101811,
 'gamma': 1.5444895385933148,
 'reg_lambda': 0.24970737145052724}

## 4. Run the Training Pipeline

For this demonstration, we do not overwrite the processed CSV files again. We pass the selected XGBoost parameters from model development to the pipeline, save the generated training artifacts in `models/`, and log the same artifacts to MLflow.

In [3]:
training_result = run_training_pipeline(
    save_processed_data=False,
    model_params=selected_xgb_params,
    model_name="XGBoost tuned",
    log_to_mlflow=True,
)

print("Model artifact:", training_result.model_path)
print("Preprocessor artifact:", training_result.preprocessor_path)
print("MLflow run ID:", training_result.mlflow_run_id)

pd.DataFrame([
    {
        "f1": training_result.metrics["f1"],
        "roc_auc": training_result.metrics["roc_auc"],
        "expected_net_value": training_result.metrics["expected_net_value"],
        "realized_net_value_for_backtest": training_result.metrics["realized_net_value_for_backtest"],
    }
]).round(4)

Model artifact: /home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master/models/20260618_115204_XGBClassifier_model.joblib
Preprocessor artifact: /home/cbaldermann/Projekte/ADS_II_MLOPS/2026-06-ads-II-master/models/20260618_115204_preprocessor.joblib
MLflow run ID: 1a77e9feb1464cfb96772348999b15cf


,f1,roc_auc,expected_net_value,realized_net_value_for_backtest
0,0.5913,0.8467,2789.7773,3000.0


## 5. Simulate New Inference Data

For demonstration, we take a few rows from the raw dataset and remove the target column. This simulates new customer observations where `Churn` is not known yet.

In [4]:
raw_sample = pd.read_csv(RAW_DATA_DIR / "Telco-Customer-Churn.csv").head(10)
inference_input = raw_sample.drop(columns=["Churn"])

inference_input.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


## 6. Run the Inference Pipeline

The inference pipeline loads the saved preprocessor and saved model from their local artifact paths. In a larger setup, these artifacts could also be retrieved from the MLflow artifact store or a model registry.

In [5]:
predictions = run_inference_pipeline(
    input_data=inference_input,
    model_path=training_result.model_path,
    preprocessor_path=training_result.preprocessor_path,
    threshold=0.5,
)

predictions

,customerID,churn_probability,churn_prediction
0,7590-VHVEG,0.715075,1
1,5575-GNVDE,0.047393,0
2,3668-QPYBK,0.336025,0
3,7795-CFOCW,0.043585,0
4,9237-HQITU,0.669073,1
5,9305-CDSKC,0.823422,1
6,1452-KIOVK,0.457112,0
7,6713-OKOMC,0.194941,0
8,7892-POOKP,0.613804,1
9,6388-TABGU,0.037899,0


## 7. What This Solves

This structure reduces a common production risk: training and inference accidentally using different preprocessing logic.

The training pipeline fits and saves the preprocessor, then logs it together with the model to MLflow. The inference pipeline loads that exact preprocessor. This means that medians, category handling, scaling parameters, and encoded feature order are reused consistently.

This is a manual, reproducible training workflow. It is enough for a small project where artifacts are reviewed and then manually moved into a cloud or serving environment.

The same structure is also a building block for Continuous Training, but it is not a full Continuous Training setup yet. A CT setup would need additional automation and controls: a trigger such as a schedule, new data, or a drift signal; data validation; comparison against the current production model; quality gates; model registry stages; approval rules; deployment logic; monitoring; and rollback options.

The next step in this course project is therefore not full CT. The next step is to make the pipeline easier to run, test the important pieces, and then use GitHub Actions as a lightweight CI demonstration.

## 8. Run the Pipeline from the Terminal

The same pipeline can now be executed without opening a notebook. Start from the repository root:

```bash
python -m src.cli train \
  --params-path reports/model_results/05_selected_xgboost_params.json \
  --no-save-processed-data \
  --log-to-mlflow
```

This trains the selected model configuration, saves the model and preprocessor artifacts in `models/`, and logs metrics plus artifacts to MLflow.

Batch inference can also be run from the terminal once model and preprocessor paths are known:

```bash
python -m src.cli predict \
  data/raw/new_customers.csv \
  models/<timestamp>_XGBClassifier_model.joblib \
  models/<timestamp>_preprocessor.joblib \
  reports/model_results/new_customer_predictions.csv
```

This is still a lightweight local command-line interface, not a full orchestration system. Its purpose is to make the workflow reproducible and callable from tests or GitHub Actions.